<a href="https://colab.research.google.com/github/whiplaaash/TF-NLP/blob/main/colab_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to NLP fundamentals in Tensor Flow

NLP has the goal of deriving information out of natural language (could be sequences, text or speach).

Another common term for NLP problems is sequence to seqence (seq2seq).

# Check for GPU (to change from CPU to GPU use "Srodowisko wykonawcze" and then "Zmien typ srodowiska wykonawczego").


In [1]:
!nvidia-smi -L

/bin/bash: line 1: nvidia-smi: command not found


# Get helper functions

In [2]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py

--2026-11-12 20:29:50--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-11-12 20:29:50 (81.6 MB/s) - ‘helper_functions.py’ saved [10246/10246]



# Import a series of helper functions for the notebook.

In [3]:
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

# Get a dataset

We are going to be using a dataset taken from Kaggle. It is an introduction to NLP dataset (text samples of Tweets labelled as a disaster or not a disaster).

Se the original source:
https://www.kaggle.com/competitions/nlp-getting-started

In [4]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip


--2026-11-12 20:30:06--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 108.177.98.207, 74.125.195.207, 172.253.117.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|108.177.98.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.01s   

2026-11-12 20:30:06 (47.6 MB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



In [5]:
# Unzip data
unzip_data("nlp_getting_started.zip")

# Visualizing a text dataset

In [6]:
# We use pandas package to read the data.

import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df.head()


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [7]:
train_df["text"][0]

'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'

In [8]:
# Shuffle training dataframe

train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [9]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [10]:
# How many examples of class are there?
train_df_shuffled.target.value_counts()

# Data are quite well balanced (60:40)

,count
target,
0,4342
1,3271


In [11]:
# How many total samples?
len(train_df), len(test_df)

(7613, 3263)

# Let's visualize some data

In [12]:
# Let's generate 10 random samples
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[["text", "target"]][random_index:random_index+5].itertuples():
# we generate random samples in the range [random_index:random_index+5]
# itertuples() - returns values in the row as a tuple
  _, text, target = row
  print(f"Target: {target}", "(real disaster)" if target==1 else "(not real disaster)")
  print(f"Text:\n{text}\n")
  print("-----\n")

Target: 1 (real disaster)
Text:
My last two weather pics from the storm on August 2nd. People packed up real fast after the temp dropped and winds picked up.

-----

Target: 1 (real disaster)
Text:
17 people displaced after 3-alarm fire tore through 2 apartment buildings on Second Street in Manchester. -- http://t.co/NzqwTCYidv #MHT

-----

Target: 1 (real disaster)
Text:
Westminister Sr22 W / Knott St **Trfc Collision-No Inj** http://t.co/EUmlca1Edw

-----

Target: 1 (real disaster)
Text:
The Fake of Nanking Massacre-4 Eyewitnesses (English): http://t.co/TiPnDEmPuz  #Obama #Clinton #Bush #GOP #ABC #CBS #BBC #CNN #WSJ #WPO

-----

Target: 0 (not real disaster)
Text:
How would I have survived without the mute option on whatsapp!

-----



In [13]:
# Split data into training and validation sets
from sklearn.model_selection import train_test_split

train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled["text"].to_numpy(),
                                                                            train_df_shuffled["target"].to_numpy(),
                                                                            test_size=0.1, # use 10% of data as validation set
                                                                            random_state=42 # for reproduction of the data
                                                                            )

In [14]:
# Check lenghts
len(train_sentences), len(val_sentences), len(train_labels), len(val_labels)

(6851, 762, 6851, 762)

In [15]:
# check the first 10 samples
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

# Converting text to numbers (tokenization and embbeding)

Before building a model based on words, the first thing to do is to convert words to numbers.

**Tokenization**: converting tokens (words, subwords, characters) into numbers, usually with one-hot encoding.

**Embedding**: takes into account the context of w word, probability of beng close to other words. We may use pre-learnt embedding. For each token you create a matrix of feature vectors.

# Tokenization (text vectorization)

In [16]:
train_sentences[:5]

array(['@mogacola @zamtriossu i screamed after hitting tweet',
       'Imagine getting flattened by Kurt Zouma',
       '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
       "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
       'Somehow find you and I collide http://t.co/Ee8RpOahPk'],
      dtype=object)

In [17]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Use the default parameters
text_vectorizer = TextVectorization(max_tokens=None, # how many tokens in the whole set of vocabulary
                                    standardize="lower_and_strip_punctuation",
                                    split="whitespace",
                                    ngrams=None, # groups of n number of words
                                    output_mode="int", # how to represent words as numbers
                                    output_sequence_length=None, # how long do you want your sequences to be
                                    pad_to_max_tokens=False)

In [18]:
# Find the avergae number of tokens in the training tweets
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [19]:
# Setup text vectorization variables
max_vocab_length = 10000 # max number of words to have in our vocabulary
max_length = 15 # max length our sentences will be

# Use fixed, and no default, parameters
text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [20]:
# Fit the text vectorizer to the training data
text_vectorizer.adapt(train_sentences)

In [21]:
# Create a sample sentence and tokenize it
sample_sentence = "There is a flood on my street!"
text_vectorizer([sample_sentence])


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[ 74,   9,   3, 232,  11,  13, 698,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [22]:
# choose a random sentence from the training dataset and tokenize it
random_sentence = random.choice(train_sentences)
print(f"Original sentence:\n{random_sentence}\
      \n\nVectorized version:")
text_vectorizer([random_sentence])

Original sentence:
Stepped outside with a drink and a cigarette and immediately locked eyes with a jogger. Worlds really do collide.      

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[3381,  676,   14,    3, 1530,    7,    3, 4007,    7, 5385, 5201,
         708,   14,    3,    1]])>

In [23]:
# Get the unique words in vocabulary
words_in_vocab = text_vectorizer.get_vocabulary() # get all the unique words in the vocabulary
top_5_words = words_in_vocab[:5] # get the most common words
bottom_5_words = words_in_vocab[-5:] # get the least common words in vocab
print(f"Number of words in vocabulary: {len(words_in_vocab)}")
print(f"5 most common words: {top_5_words}")
print(f"5 least common words: {bottom_5_words}")


Number of words in vocabulary: 10000
5 most common words: ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
5 least common words: [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


# Embedding

To make an embedding we are going to use Tensor Flow embedding layer.

The parameters we care about most:
- `input_dim` - how many words are in vocabulary
- `output_dim` - the size of the output embedding vector, for example: a numbe 100 means that each token will be represented by a vector 100 long
- `input_length` - length of the sequences being passed to the embedding layer (this parameter no longer exist!)

In [24]:
from tensorflow.keras import layers
embedding = layers.Embedding(name="Embedding",
                             input_dim=max_vocab_length, #set the input shape
                             output_dim=128 # something divisible by 8, output shape
                             )
embedding

<Embedding name=Embedding, built=False>

In [25]:
# Get a random sentence from the trainig set
random_sentence = random.choice(train_sentences)
print(f"Original sentence:\n{random_sentence}\
\n\nEmbedded version:")

# Embed a random sentence (turn it into dense vectors of fixed size)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original sentence:
Twin Storms Blow Through Calgary ~ 1 http://t.co/rzaGDnWTAH http://t.co/uqEttaGSCU

Embedded version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.03153875, -0.0397122 , -0.01483013, ...,  0.00589992,
          0.02942554, -0.00360268],
        [-0.03584198,  0.02160515,  0.03396125, ...,  0.0148269 ,
         -0.01016168, -0.0057288 ],
        [-0.02105831, -0.03258053, -0.00205637, ..., -0.01815014,
         -0.01722096, -0.03019781],
        ...,
        [-0.01238449,  0.03725595,  0.01229285, ..., -0.02822943,
         -0.04499974,  0.03835981],
        [-0.01238449,  0.03725595,  0.01229285, ..., -0.02822943,
         -0.04499974,  0.03835981],
        [-0.01238449,  0.03725595,  0.01229285, ..., -0.02822943,
         -0.04499974,  0.03835981]]], shape=(1, 15, 128), dtype=float32)>

In [26]:
# Check out a single token's embedding
sample_embed[0][0], sample_embed[0][0].shape, random_sentence[0]

# This array represents the first word in the random sentence
# The array shows random numbers which will be updated when the model learns
# The length of each suach array is 128 and is fixed

(<tf.Tensor: shape=(128,), dtype=float32, numpy=
 array([ 0.03153875, -0.0397122 , -0.01483013, -0.01104398, -0.02638054,
        -0.03128188, -0.01499422,  0.04523266,  0.04371196,  0.0118705 ,
         0.04135476, -0.0307137 ,  0.0457402 ,  0.01489856,  0.01452829,
         0.01342196, -0.00939686, -0.02745522,  0.04690823,  0.0017385 ,
         0.04794592,  0.02040691,  0.02399525,  0.00939571,  0.03931463,
         0.02269704,  0.03135843,  0.0052761 ,  0.00761753, -0.0168388 ,
         0.04654069,  0.02351088,  0.0164136 ,  0.00533814, -0.03608189,
         0.00650029, -0.02922835, -0.00012463, -0.00115801,  0.03153436,
        -0.0306941 ,  0.04024055,  0.03531152, -0.00189269, -0.02316746,
        -0.02059375, -0.00549598,  0.00146595,  0.03900496,  0.01749067,
        -0.00694873,  0.03653747,  0.02382532, -0.01048368, -0.04809085,
        -0.0034501 , -0.00032562,  0.0281546 , -0.04161601, -0.01250724,
        -0.0438166 , -0.04333004,  0.02677004, -0.04608228,  0.03935735,
  

# Modelling a text dataset (running a series of experiments)

After turning a text into numbers we will be running 8 experiments. We will start with a baseline and will be improving it.

* Model 0: Naive Bayes
* Model 1: Feed-forward neural network (dense model)
* Model 2: LSTM model (RNN)
* Model 3: GRU model (RNN)
* Model 4: Bidirectional LSTM model (RNN)
* Model 5: 1D Convolutional Neural Network model (CNN)
* Model 6: Tensor Flow Hub Pretrined Feature Extractor (using transfer learning for NLP)
* Model 7: Same as model 6 but with 10% of training data

To do all this we will go through the standard procedure of machine  learning:
- create a model
- build a model
- fit a model
- evaluate a model


# Model 0: Naive Bayes

This is a non-deep-learning model. It is part of Machine Learning and it comes from sklearn package.

⭕ It is important to start with a baseline model and then improve it. We will start with a Multinomial Naive Bayes with TF-IDF formula.

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [28]:
# Create tokanization and modelling pipeline
model_0 = Pipeline([
    ("tfidf", TfidfVectorizer()), # convert words to numbers using tfidf
    ("clf", MultinomialNB()) # model the text, "clf" means a classifier
])

# Fir the pipeline to the training data
model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

In [29]:
# Evaluate our baseline model
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Our baseline model achieves an accuracy of: {baseline_score*100:.2f}%")

Our baseline model achieves an accuracy of: 79.27%


This accuracy is more than a guess, which would be around 55%.

In [30]:
# Make predictions
baseline_preds = model_0.predict(val_sentences)
baseline_preds[:20]

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1])

In [31]:
train_labels

array([0, 0, 1, ..., 1, 1, 0], shape=(6851,))

# Create a function which will be tracking the most important metrics across all models

Metrics:
- accuracy (not the best metric for imbalanced classes)
- precision (higher precison leads to less false positives)
- recall (higher recall leads to less false negatives)
- F1 score (combination of precision and recall)

In [32]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# similarly one could use tensorflow metrics instead of sklearn metrics

def calculate_results(y_true, y_pred):
  """
  Calculates model accuracy, precision, recall, and f1-score of a binary classification model.
  """
  # Calculate model accuracy:
  model_accuracy = accuracy_score(y_true, y_pred)*100
  # Calculate model precision, recall, f1-score using weighted average (weighted by support, that is, the number of cases in each class)
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
  # the empty underscore "_" above stands for the support, but we are not interested in it
  model_results = {
      "accuracy": model_accuracy,
      "precision": model_precision,
      "recall": model_recall,
      "f1-score": model_f1}
  return model_results


In [33]:
# Get baseline results
baseline_results = calculate_results(y_true=val_labels, y_pred=baseline_preds)
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1-score': 0.7862189758049549}

# Model 1: Simple dense model

⭐ Reminder: **callbacks** run some code at specific points of the code to do some extra operations on top of the running model:
- stop training when performance is not improving, [EarlyStopping],
- saving model during training [ModleCheckpoint],
- log metrics for visualization [TensorBoard],
- lower the training rate when progress stalls [ReduceLROnPlateau],
- change learning rate over time [LearningRateScheduler]

In [34]:
# Create a tensorboard callback (need to create a new one for each model)
from helper_functions import create_tensorboard_callback

# Create a directory to save TensorBoard logs
SAVE_DIR = "model_logs"

# Build model with the Functional API
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype=tf.string) # inputs are 1D strings
x = text_vectorizer(inputs) # turn the input text into numbers
x = embedding(x) # create an embedding of the numberized inputs
x = layers.GlobalAveragePooling1D()(x) # this converts a sequence into a single vector so that ndim of predictions and labels is the same
# pooling makes the model predicts one value per sentence, and per token
outputs = layers.Dense(1, activation="sigmoid")(x) # create an output layer, we want binary outputs so we use sigmoid activation function

model_1 = tf.keras.Model(inputs, outputs, name="model_1_dense")


In [35]:
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,280,129 (4.88 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
# Compile the model
model_1.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [37]:
# Fit the model
model_1_history = model_1.fit(x=train_sentences,
                              y=train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                         experiment_name="model_1_dense")])

Saving TensorBoard log files to: model_logs/model_1_dense/20261112-203011
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.6929 - loss: 0.6051 - val_accuracy: 0.7441 - val_loss: 0.5360
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.8192 - loss: 0.4413 - val_accuracy: 0.7835 - val_loss: 0.4729
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.8612 - loss: 0.3477 - val_accuracy: 0.7887 - val_loss: 0.4656
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8899 - loss: 0.2850 - val_accuracy: 0.7900 - val_loss: 0.4672
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9105 - loss: 0.2389 - val_accuracy: 0.7927 - val_loss: 0.4831


In [38]:
# check the results
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7927 - loss: 0.4831


[0.4830971956253052, 0.7926509380340576]

In [39]:
# Make some predictions and evaluate those
model_1_pred_probs = model_1.predict(val_sentences)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [40]:
model_1_pred_probs.shape

(762, 1)

In [41]:
# Look at the single prediction
model_1_pred_probs[0]

array([0.3144946], dtype=float32)

In [42]:
# Look at the first 10 predictions
model_1_pred_probs[:10]

array([[0.3144946 ],
       [0.82710016],
       [0.9975336 ],
       [0.09984806],
       [0.10948028],
       [0.93941325],
       [0.90045726],
       [0.99301684],
       [0.9597369 ],
       [0.24447429]], dtype=float32)

In [43]:
# To compare our preds to the val labels wee need to first convert them to the val format, which is to have 0 and 1 values
# First we squeeze the preds to cancel one extra dimension
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0.], dtype=float32)>

In [44]:
# Calculate our model_1 results using the function that we defined before
model_1_results = calculate_results(y_true=val_labels, y_pred=model_1_preds)
model_1_results

{'accuracy': 79.26509186351706,
 'precision': 0.7997486438518866,
 'recall': 0.7926509186351706,
 'f1-score': 0.789139453229987}

In [45]:
# Comapare with the baseline results
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1-score': 0.7862189758049549}

In [46]:
import numpy as np
np.array(list(model_1_results.values())) > np.array(list(baseline_results.values()))

array([False, False, False,  True])

It appears that the baseline results are better than the results coming from the dense neural network trainig.

# Visualizing learned embeddings

In [47]:
# Get the vocabulary from the text vectorization layer
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

(10000,
 ['',
  '[UNK]',
  np.str_('the'),
  np.str_('a'),
  np.str_('in'),
  np.str_('to'),
  np.str_('of'),
  np.str_('and'),
  np.str_('i'),
  np.str_('is')])

In [48]:
max_vocab_length

10000

In [49]:
# Model_1 summary
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,840,389 (14.65 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,560,260 (9.77 MB)

In [51]:
# Get the weigth matrix of embedding layer
# (these are numerical representations of each token in our training data)
embed_weights = model_1.get_layer("Embedding").get_weights()[0]
embed_weights
print(embed_weights.shape) # same sa vocab size and embedding dim (output dim of our embedding layer)

(10000, 128)


Now we have got the embedding matrix our model has learned to represent our tokens. Let's see how we can visulaize it.

To do so, TensorFlow has a handy tool caller projector: https://projector.tensorflow.org

And TensorFlow also has an incredible guide on word embeddings themselves: https://www.tensorflow.org/text/guide/word_embeddings?hl=pl



In [52]:
# Create embedding files (we got this from TensorFlow word embedding documentation)
import io

out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(words_in_vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = embed_weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()

In [53]:
# Download fiels from Colab to upload to projector
try:
  from google.colab import files
  files.download('vectors.tsv')
  files.download('metadata.tsv')
except Exception:
  pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

At this point you can upload the files to:
https://projector.tensorflow.org
to see the distribution of the weights (of words in vocab) in 3D.

You can see more on embeddings here:
- https://jalammar.github.io/illustrated-word2vec/


# Recurrent Neural Networks (RNN)

RNNs are useful for sequence data.

The premise of RNNs is to use the representation of previous input to aid the representation of later input.

Nice overview of RNNs:
- https://youtu.be/qjrad0V0uJE (from MIT)
- https://colah.github.io/posts/2015-08-Understanding-LSTMs/ (from Chris Olah)
- https://karpathy.github.io/2015/05/21/rnn-effectiveness/ (unreasonable effectiveness of RNNs from Andrej Karpathy)



# Model 2: LSTM (Long-Short Term Memory - one of the most popular RNNs)

Structure:
Input (text) -> Tokenize -> Embedding -> Layers (RNNs/Dense) -> Output (label probability)

In [55]:
# Create an LSTM model
from tensorflow.keras import layers
inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)
#print(x.shape)
#x = layers.LSTM(64, return_sequences=True)(x) # when you are stacking layers: in the first lstm layer you need to set return_sequences=True to keep correct dim
#print(x.shape)
x = layers.LSTM(64)(x)
#print(x.shape)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model_2 = tf.keras.Model(inputs, outputs, name="model_2_LSTM")

In [56]:
model_2.summary()

Model: "model_2_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,333,633 (5.09 MB)

 Trainable params: 1,333,633 (5.09 MB)

 Non-trainable params: 0 (0.00 B)

In [57]:
# Compile the model
model_2.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [59]:
# Fit the model
model_2_history = model_2.fit(train_sentences,
                              train_labels,
                              epochs=5,
                              validation_data=(val_sentences, val_labels),
                              callbacks=[create_tensorboard_callback(SAVE_DIR, "model_2_lstm")])

Saving TensorBoard log files to: model_logs/model_2_lstm/20261112-204553
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - accuracy: 0.9406 - loss: 0.1570 - val_accuracy: 0.7795 - val_loss: 0.6170
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - accuracy: 0.9507 - loss: 0.1288 - val_accuracy: 0.7835 - val_loss: 0.8593
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - accuracy: 0.9569 - loss: 0.1067 - val_accuracy: 0.7743 - val_loss: 0.7802
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - accuracy: 0.9651 - loss: 0.0869 - val_accuracy: 0.7795 - val_loss: 1.0849
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 58ms/step - accuracy: 0.9682 - loss: 0.0759 - val_accuracy: 0.7795 - val_loss: 1.2872


In [60]:
# Make prdictions
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step


array([[3.81907851e-01],
       [7.44076014e-01],
       [9.99957681e-01],
       [1.41123645e-02],
       [1.38697005e-05],
       [9.99726892e-01],
       [9.35499489e-01],
       [9.99988496e-01],
       [9.99953210e-01],
       [6.26564801e-01]], dtype=float32)

In [62]:
# Convert pred probs to labels
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [63]:
# Calculate results
model_2_results = calculate_results(y_true=val_labels,
                                    y_pred=model_2_preds)
model_2_results

{'accuracy': 77.95275590551181,
 'precision': 0.7797634583030512,
 'recall': 0.7795275590551181,
 'f1-score': 0.7784163768415736}

In [64]:
baseline_results

{'accuracy': 79.26509186351706,
 'precision': 0.8111390004213173,
 'recall': 0.7926509186351706,
 'f1-score': 0.7862189758049549}